# 04 — Preprocessing and Feature Decisions

## Objective

In this notebook, I prepare the Telco churn dataset for the first modeling stage.

The goal is not only to clean the data, but also to make clear feature decisions before training any model.

At this stage, I focus on:

- standardizing missing-like values
- deciding which columns should not be used as model inputs
- identifying possible leakage-sensitive columns
- reviewing overlapping or redundant features
- creating a cleaner baseline dataset for modeling

I want the baseline dataset to be simple enough to explain, but still informative enough for the first model comparison.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path
import json

In [3]:
DATA_PATH = Path("../data/raw/telco.csv")

df = pd.read_csv(DATA_PATH)
df.head()

,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
display(pd.DataFrame({"column": df.columns}))

Shape: (7043, 50)

Columns:


,column
0,Customer ID
1,Gender
2,Age
3,Under 30
4,Senior Citizen
5,Married
6,Dependents
7,Number of Dependents
8,Country
9,State


## 1. Define target variable

For this project, I use `Churn Label` as the target variable.

This column directly tells whether a customer churned or not, so it is the clearest outcome column for binary classification.

I map the values into a numerical target:

- `No` → 0
- `Yes` → 1

This makes the target ready for scikit-learn models.

In [5]:
df["target_churn"] = df["Churn Label"].map({"No": 0, "Yes": 1})

df[["Churn Label", "target_churn"]].sample(7)

,Churn Label,target_churn
3739,No,0
4389,No,0
4943,No,0
6322,No,0
1342,Yes,1
5627,No,0
1750,Yes,1


In [6]:
print(df["Churn Label"].value_counts(dropna=False))
print()
print(df["target_churn"].value_counts(dropna=False))

Churn Label
No     5174
Yes    1869
Name: count, dtype: int64

target_churn
0    5174
1    1869
Name: count, dtype: int64


## 2. Create a working modeling dataframe

Before making preprocessing changes, I create a working copy of the raw dataset.

I do this because I want to keep the original dataframe unchanged and make all modeling-related decisions on a separate copy.

This makes the workflow safer and easier to trace.

In [7]:
model_df = df.copy()
model_df.shape

(7043, 51)

## 3. Normalize missing-like values

The earlier audit and SQL validation showed that missing information is not stored in one consistent way.

Some columns contain actual missing values, while others use the literal text `"None"`.

If I leave these values as they are, the model may treat `"None"` as a real category instead of missing or business-specific information.

For that reason, I first standardize missing-like values before making column-specific decisions.

In [8]:
object_cols = model_df.select_dtypes(include="object").columns.tolist()
object_cols

['Customer ID',
 'Gender',
 'Under 30',
 'Senior Citizen',
 'Married',
 'Dependents',
 'Country',
 'State',
 'City',
 'Quarter',
 'Referred a Friend',
 'Offer',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Internet Type',
 'Online Security',
 'Online Backup',
 'Device Protection Plan',
 'Premium Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Streaming Music',
 'Unlimited Data',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Customer Status',
 'Churn Label',
 'Churn Category',
 'Churn Reason']

In [9]:
model_df[object_cols] = model_df[object_cols].replace("None", np.nan)
model_df[object_cols] = model_df[object_cols].replace(r"^\s*$", np.nan, regex=True)

In [10]:
review_semantic_cols = ["Offer", "Internet Service", "Internet Type"]

for col in review_semantic_cols:
    if col in model_df.columns:
        print(f"\n--- {col} ---")
        print(model_df[col].value_counts(dropna=False))

if {"Internet Service", "Internet Type"}.issubset(model_df.columns):
    display(
        pd.crosstab(
            model_df["Internet Service"].fillna("Missing"),
            model_df["Internet Type"].fillna("Missing"),
            margins=True
        )
    )


--- Offer ---
Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64

--- Internet Service ---
Internet Service
Yes    5517
No     1526
Name: count, dtype: int64

--- Internet Type ---
Internet Type
Fiber Optic    3035
DSL            1652
NaN            1526
Cable           830
Name: count, dtype: int64


Internet Type,Cable,DSL,Fiber Optic,Missing,All
Internet Service,,,,,
No,0,0,0,1526,1526
Yes,830,1652,3035,0,5517
All,830,1652,3035,1526,7043


In [11]:
if {"Internet Service", "Internet Type"}.issubset(model_df.columns):
    no_internet_mask = model_df["Internet Service"].eq("No") & model_df["Internet Type"].isna()
    model_df.loc[no_internet_mask, "Internet Type"] = "No Internet"

if "Internet Type" in model_df.columns:
    model_df["Internet Type"] = model_df["Internet Type"].fillna("Missing")

In [12]:
pd.crosstab(
    model_df["Internet Service"].fillna("Missing"),
    model_df["Internet Type"].fillna("Missing"),
    margins=True
)

Internet Type,Cable,DSL,Fiber Optic,No Internet,All
Internet Service,,,,,
No,0,0,0,1526,1526
Yes,830,1652,3035,0,5517
All,830,1652,3035,1526,7043


### Internet Type Missingness Review

I reviewed `Internet Type` together with `Internet Service` to understand whether the missing values were random.

The check showed that missing `Internet Type` values belong to customers whose `Internet Service` is `No`.

So these blanks do not mean “unknown internet type”. They mean that the customer does not have internet service.

Because of that, I recoded these values as `No Internet`.

This is a small cleaning step, but it is important because the replacement is based on business meaning, not only on a technical missing-value rule.

In [13]:
if "Offer" in model_df.columns:
    print(model_df["Offer"].value_counts(dropna=False))

Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64


In [14]:
if "Offer" in model_df.columns:
    model_df["Offer"] = model_df["Offer"].fillna("No Offer")

In [15]:
model_df["Offer"].value_counts(dropna=False)

Offer
No Offer    3877
Offer B      824
Offer E      805
Offer D      602
Offer A      520
Offer C      415
Name: count, dtype: int64

### Offer Missingness Review

The `Offer` column contains many missing values, while the non-missing values are limited to named offer categories such as `Offer A` to `Offer E`.

This pattern suggests that the blank values probably mean that no offer was assigned to the customer.

Because of that, I recoded missing values in `Offer` as `No Offer`.

Again, I am not treating this as a generic missing-value problem. I am using the business meaning of the column to make a clearer preprocessing decision.

In [16]:
missing_summary = (
    model_df.isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
missing_summary.columns = ["column", "missing_count"]

missing_summary[missing_summary["missing_count"] > 0]

,column,missing_count
0,Churn Reason,5174
1,Churn Category,5174


## 4. Remove direct churn outcome columns

Some columns are directly related to the churn result or describe what happened after the customer churned.

These columns should not be used as model inputs because they can leak the answer into the model.

Columns removed at this stage:

- `Customer Status`
- `Churn Category`
- `Churn Reason`
- `Churn Score`

The model should learn from customer profile, contract, service, and billing information — not from columns that already summarize the churn outcome.

In [17]:
leakage_cols = [
    "Customer Status",
    "Churn Category",
    "Churn Reason",
    "Churn Score"
]

existing_leakage_cols = [col for col in leakage_cols if col in model_df.columns]
existing_leakage_cols

['Customer Status', 'Churn Category', 'Churn Reason', 'Churn Score']

## 5. Remove identifier and location columns

Some columns are useful for identifying or describing customers, but they are not suitable as direct model features.

For example, `Customer ID` is unique for each customer. It can be useful for lookup and joining data, but it should not be used by the model to learn churn behavior.

I also removed detailed location columns such as city, zip code, latitude, and longitude from the baseline model.

At this stage, I want the first modeling dataset to stay clean and focused on customer behavior and service-related features.

Removed columns include:

- `Customer ID`
- `Country`
- `State`
- `Count`
- `Lat long`
- `CLTV`
- `Total Revenue`
- `City`
- `Zip Code`
- `Latitude`
- `Longitude`
- `Quarter`

In [18]:
review_cols = [
    "Customer ID",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",
    "Quarter",
    "CLTV",
    "Total Revenue"
]

existing_review_cols = [col for col in review_cols if col in model_df.columns]
existing_review_cols

['Customer ID',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Latitude',
 'Longitude',
 'Quarter',
 'CLTV',
 'Total Revenue']

## 6. Review derived and overlapping demographic columns

Some demographic columns contain overlapping information.

For example:

- `Age` and `Under 30`
- `Age` and `Senior Citizen`
- `Number of Dependents` and `Dependents`
- `Number of Referrals` and `Referred a Friend`

To keep the baseline model cleaner, I kept the more detailed numerical versions where possible.

This makes the model input less repetitive and easier to interpret later.

In [19]:
candidate_redundant_cols = [
    "Age",
    "Under 30",
    "Senior Citizen",
    "Dependents",
    "Number of Dependents",
    "Referred a Friend",
    "Number of Referrals",
    "Monthly Charge",
    "Tenure in Months",
    "Total Charges"
]

existing_redundant_cols = [col for col in candidate_redundant_cols if col in model_df.columns]
existing_redundant_cols

['Age',
 'Under 30',
 'Senior Citizen',
 'Dependents',
 'Number of Dependents',
 'Referred a Friend',
 'Number of Referrals',
 'Monthly Charge',
 'Tenure in Months',
 'Total Charges']

## 7. Drop non-feature columns for the baseline modeling dataset

At this point, the baseline dataset is created by removing columns that fall into one of three groups:

- leakage-sensitive fields
- columns set aside during review
- overlapping features removed for interpretability

This gives a cleaner starting point for the first modeling round.

## Column Exclusion Rationale

Columns were removed from the baseline modeling dataset for practical reasons such as:

- leakage risk
- identifier-only role
- limited analytical value
- overly detailed geographic information
- business metrics that need separate review
- overlapping representation of features already retained

The goal here is not to create the final perfect dataset, but to build a cleaner and more interpretable baseline version for modeling.

In [20]:
drop_reasons = pd.DataFrame({
    "column": [
        "Customer Status",
        "Churn Category",
        "Churn Reason",
        "Churn Score",
        "Customer ID",
        "Count",
        "Country",
        "State",
        "City",
        "Zip Code",
        "Lat Long",
        "Latitude",
        "Longitude",
        "Quarter",
        "CLTV",
        "Total Revenue",
        "Under 30",
        "Senior Citizen",
        "Dependents",
        "Referred a Friend"
    ],
    "reason": [
        "Potential post-outcome status; leakage risk",
        "Post-churn categorical information; direct leakage",
        "Post-outcome explanatory field; direct leakage",
        "Derived churn-risk score; high leakage risk",
        "Unique identifier; not generalizable",
        "Constant / low analytical value",
        "Single-country field; low analytical value",
        "High-cardinality geographic detail",
        "High-cardinality geographic detail",
        "High-cardinality geographic detail",
        "High-cardinality geographic detail",
        "High-cardinality geographic detail",
        "High-cardinality geographic detail",
        "Temporal grouping field requiring separate review",
        "Derived business metric requiring further review",
        "Derived business metric requiring further review",
        "Derived age-group flag overlapping with Age",
        "Derived age-group flag overlapping with Age",
        "Binary indicator overlapping with Number of Dependents",
        "Binary indicator overlapping with Number of Referrals"
    ]
})

drop_reasons = drop_reasons[drop_reasons["column"].isin(model_df.columns)].copy()
drop_reasons

,column,reason
0,Customer Status,Potential post-outcome status; leakage risk
1,Churn Category,Post-churn categorical information; direct lea...
2,Churn Reason,Post-outcome explanatory field; direct leakage
3,Churn Score,Derived churn-risk score; high leakage risk
4,Customer ID,Unique identifier; not generalizable
6,Country,Single-country field; low analytical value
7,State,High-cardinality geographic detail
8,City,High-cardinality geographic detail
9,Zip Code,High-cardinality geographic detail
11,Latitude,High-cardinality geographic detail


In [21]:
report_path = "../reports/drop_reasons.csv"
drop_reasons.to_csv(report_path, index=False)

print(f"Feature exclusion rationale saved as report: {report_path}")

Feature exclusion rationale saved as report: ../reports/drop_reasons.csv


In [22]:
drop_cols = (
    existing_leakage_cols
    + existing_review_cols
    + [col for col in ["Under 30", "Senior Citizen", "Dependents", "Referred a Friend"] if col in model_df.columns]
    + ["Churn Label"]
)

drop_cols = [col for col in drop_cols if col in model_df.columns]

baseline_df = model_df.drop(columns=drop_cols).copy()

print("Dropped columns:")
print(drop_cols)

print("\nBaseline shape:", baseline_df.shape)

Dropped columns:
['Customer Status', 'Churn Category', 'Churn Reason', 'Churn Score', 'Customer ID', 'Country', 'State', 'City', 'Zip Code', 'Latitude', 'Longitude', 'Quarter', 'CLTV', 'Total Revenue', 'Under 30', 'Senior Citizen', 'Dependents', 'Referred a Friend', 'Churn Label']

Baseline shape: (7043, 32)


In [23]:
baseline_df.columns.tolist()

['Gender',
 'Age',
 'Married',
 'Number of Dependents',
 'Population',
 'Number of Referrals',
 'Tenure in Months',
 'Offer',
 'Phone Service',
 'Avg Monthly Long Distance Charges',
 'Multiple Lines',
 'Internet Service',
 'Internet Type',
 'Avg Monthly GB Download',
 'Online Security',
 'Online Backup',
 'Device Protection Plan',
 'Premium Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Streaming Music',
 'Unlimited Data',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charge',
 'Total Charges',
 'Total Refunds',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Satisfaction Score',
 'target_churn']

In [24]:
baseline_df.head()

,Gender,Age,Married,Number of Dependents,Population,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,...,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Satisfaction Score,target_churn
0,Male,78,No,0,68701,0,1,No Offer,No,0.00,...,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,3,1
1,Female,74,Yes,1,55668,1,8,Offer E,Yes,48.85,...,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,3,1
2,Male,71,No,3,47534,0,18,Offer D,Yes,11.33,...,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,2,1
3,Female,78,Yes,1,27778,1,25,Offer C,Yes,19.76,...,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2,1
4,Female,80,Yes,1,26265,1,37,Offer C,Yes,6.33,...,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,2,1


## 8. Separate features and target

After the column review is complete, the baseline dataset is split into `X` and `y` for the next modeling steps.

In [25]:
X = baseline_df.drop(columns=["target_churn"])
y = baseline_df["target_churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 31)
y shape: (7043,)


In [26]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 13
Categorical features: 18


In [27]:
print("Numeric columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)

Numeric columns:
['Age', 'Number of Dependents', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Satisfaction Score']

Categorical columns:
['Gender', 'Married', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method']


## 9. Final preprocessing notes

At the end of this preprocessing stage, the dataset is in a much cleaner state for baseline modeling.

Missing-like values have been standardized, semantically meaningful blanks have been recoded, leakage-prone columns have been removed, and a few overlapping indicators have been simplified to make the model easier to interpret.

Two features are intentionally kept for later review:

- `Satisfaction Score`, which will be tested again in a more conservative no-leakage comparison
- `Total Charges`, which will be checked separately because it may overlap with `Monthly Charge` and `Tenure in Months`

The next notebook moves from feature decisions to train/test split, pipeline setup, and baseline model evaluation.

In [28]:
feature_config = {
    "target_column": "target_churn",
    "dropped_columns": drop_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}

output_path = Path("../reports/feature_config.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(feature_config, f, indent=4)

print(f"Saved to: {output_path}")

Saved to: ..\reports\feature_config.json


## Conclusion

At the end of this notebook, I have a cleaner starting point for modeling.

I am not claiming that this is the final perfect feature set yet, but the most obvious risks are now handled more carefully.

Missing-value issues were clarified, leakage-prone columns were removed, and several overlapping features were simplified.

A few variables are still being carried forward on purpose so they can be tested more carefully in later stages.

## Preprocessing Summary

In this notebook, I prepared the dataset for the first modeling stage.

The most important decisions were:

- `Churn Label` was selected as the target variable.
- Missing-like values were reviewed based on their actual meaning.
- `Internet Type` missing values were recoded as `No Internet` when the customer had no internet service.
- `Offer` missing values were recoded as `No Offer`.
- Direct churn outcome columns were removed to reduce leakage risk.
- Identifier and location columns were removed from the modeling feature set.
- Overlapping demographic columns were simplified.
- `Satisfaction Score` was marked for separate sensitivity review.
- `Total Charges` was marked for later interpretability review.
- Numerical and categorical feature groups were identified for the next preprocessing pipeline.

The output of this notebook is a cleaner and more controlled baseline dataset that can be used for model comparison.

The next step is to train and compare baseline models.